# Notebook 00 — Dataset Exploration

**NeuroDriver CNN ADAS Colombia** — academic research prototype.

Status: dataset preparation / audit stage (Phase 1). **No CNN training happens in this notebook.**

Updated 2026-09-22: the real acquired dataset is **BDD100K Images 10K**
(DatasetNinja, Supervisely polygon-annotation format), not the official
BDD100K 100k `box2d` release originally assumed — see
`docs/decisions_log.md`.


## 1. NeuroDriver context

This project is a scoped-down academic derivative of the broader NeuroDriver ADAS research effort.
NeuroDriver as a whole aims at a full advanced driver-assistance stack; this prototype isolates a
single sub-problem: **frame-level visual perception via a compact CNN**, useful as a building block
for future perception/alert features. It does not implement steering, throttle, braking, or any
CAN-level vehicle control.


## 2. Colombian problem focus

The end goal is a perception model relevant to **Colombian urban/road driving conditions**
(vehicle mix, informal traffic patterns, motorcycle density, road/infrastructure differences).
Colombian dashcam data specific to NeuroDriver is not yet integrated into this repository — see
`docs/colombian_domain_strategy.md` for the planned integration path.


## 3. Scope reduction

Out of scope for this prototype: object-detector training (YOLO/SSD/Faster R-CNN/RetinaNet/Detectron),
FastAPI/frontend implementation, mobile/Raspberry Pi deployment, TFLite/ONNX export, CAN control, and
long production training runs. In scope: frame-level classification (trained as multi-label — see
Section 6), BDD100K Images 10K as a source domain, MobileNetV2 + a lightweight CNN as KD Students, a
future ResNet50 Teacher, and Knowledge-Distillation-ready scaffolding.


## 4. BDD100K Images 10K as provisional SOURCE domain

**BDD100K is not Colombian data.** The specific package available is *BDD100K Images 10K*,
redistributed by DatasetNinja in Supervisely format (`.tar` of per-image polygon annotations) — used
here only as an initial, diverse SOURCE DOMAIN to bootstrap the CNN before Colombian NeuroDriver
TARGET DOMAIN data is available. Any distribution or performance number in this notebook describes
this BDD100K subset only. See `docs/bdd100k_setup.md`.


## 5. Colombia as future TARGET domain

The planned research progression is:

```
ImageNet MobileNetV2 -> BDD100K source-domain learning -> NeuroDriver Colombian dashcam data
-> Colombian fine-tuning -> Knowledge Distillation (ResNet50 Teacher -> MobileNetV2 / lightweight CNN Students)
-> compact Student -> web perception/alert service
```

See `docs/colombian_domain_strategy.md` for the full 9-step plan.


## 6. Four-state task, trained as multi-label

Frame-level (not object detection) target — four ADAS states for reporting:

| State | Index | Definition |
|---|---|---|
| CLEAR | 0 | no relevant vehicle and no relevant pedestrian |
| VEHICLE | 1 | >=1 relevant vehicle, no relevant pedestrian |
| PEDESTRIAN | 2 | >=1 relevant pedestrian, no relevant vehicle |
| MIXED | 3 | >=1 relevant vehicle and >=1 relevant pedestrian |

**Updated 2026-09-22:** real class counts showed severe imbalance for pure `PEDESTRIAN` frames, so
models are trained on two independent binary targets — `has_vehicle`, `has_pedestrian` — via
`BinaryCrossentropy(from_logits=True)` on two raw logits (`vehicle_logit`, `pedestrian_logit`), not a
4-way softmax. The four states above are derived post-hoc from thresholded probabilities
(`neurodriver_cnn.evaluation.metrics.labels_from_logits`) for confusion-matrix/F1/presentation
purposes only. See `docs/decisions_log.md`.

Initial category mapping: vehicle = {car, truck, bus, motorcycle}; pedestrian = {pedestrian};
auxiliary (excluded from vehicle/pedestrian) = {rider, bicycle}.


## 7. Motorcycle handling policy

Motorcycles count toward `has_vehicle=True` in the multi-label targets, but are tracked explicitly via
`has_motorcycle` / `num_motorcycles` fields in the manifest so a **motorcycle-containing-frame
evaluation slice** remains possible later (`evaluate_motorcycle_subset`, see
`src/neurodriver_cnn/evaluation/metrics.py`). Motorcycle representation is never silently merged
away, and `rider`/`bicycle` are never silently remapped into vehicle, pedestrian, or motorcycle.


## 8. Full Frame labeling method

Every mapped object anywhere in the full image contributes to the frame label. Implemented in
`neurodriver_cnn.labeling.frame_labels.classify_fullframe`.


## 9. ADAS ROI labeling method

Only objects relevant to a configurable forward-driving-corridor Region Of Interest contribute.
A box is ROI-relevant if its center falls inside the ROI, or if
`intersection_area / bbox_area >= threshold` — **and** its `bbox_area_ratio >= min_bbox_area_ratio`
(set to `0.0005` after visual review of tiny/distant detections judged to be noise; see
`docs/decisions_log.md`, 2026-09-22). Initial configuration (see `configs/dataset_config.json`):

```json
{
  "x_min": 0.20, "x_max": 0.80,
  "y_min": 0.35, "y_max": 1.00,
  "bbox_intersection_threshold": 0.35,
  "min_bbox_area_ratio": 0.0005
}
```

Implemented in `neurodriver_cnn.labeling.frame_labels.classify_roi` /
`neurodriver_cnn.labeling.roi`. Boxes themselves are derived from Supervisely polygons via
`neurodriver_cnn.data.bdd100k.polygon_to_bbox` (`x1=min(x), y1=min(y), x2=max(x), y2=max(y)`).


In [ ]:
import sys
from pathlib import Path

# Resolve project root without hardcoding a personal path (works locally and in Colab
# once the repo is cloned/mounted).
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import json
import pandas as pd

from neurodriver_cnn.config import load_dataset_config

dataset_config = load_dataset_config(PROJECT_ROOT)
print(json.dumps(dataset_config, indent=2))


## 10. Dataset audit

Loads `reports/dataset_audit.json` (produced by `scripts/03_analyze_manifest.py`).

In [ ]:
audit_path = PROJECT_ROOT / "reports" / "dataset_audit.json"
if audit_path.exists():
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    print(json.dumps(audit, indent=2)[:3000])
else:
    audit = None
    print("PENDING: reports/dataset_audit.json not found yet.")
    print("Run scripts/00-03 after placing the BDD100K .tar (see docs/bdd100k_setup.md).")


## 11. Full Frame vs ROI distribution comparison

Real figures are generated by `scripts/03_analyze_manifest.py` once the dataset is available.

In [ ]:
from IPython.display import Image, display

for fig_name in ["class_distribution_fullframe.png", "class_distribution_roi.png", "fullframe_vs_roi.png"]:
    fig_path = PROJECT_ROOT / "reports" / "figures" / fig_name
    if fig_path.exists():
        display(Image(filename=str(fig_path)))
    else:
        print(f"PENDING: {fig_name} not generated yet.")


## 12. Visual examples

`class_examples.png` (per-class frames) and `roi_examples.png` (ROI overlay) from the same audit script.

In [ ]:
for fig_name in ["class_examples.png", "roi_examples.png", "motorcycle_distribution.png"]:
    fig_path = PROJECT_ROOT / "reports" / "figures" / fig_name
    if fig_path.exists():
        display(Image(filename=str(fig_path)))
    else:
        print(f"PENDING: {fig_name} not generated yet.")


## 13. Splits and leakage prevention

Rules enforced by `neurodriver_cnn.data.manifest` and `scripts/05_validate_dataset.py`:

- split assigned **before** any augmentation;
- augmentation applied to TRAIN only (see Notebook 01);
- TEST is never used for tuning;
- splitting is **group-aware** (`group_aware_split`), but this dataset has no real sequence/video
  identifier (independent frames), so it falls back to its documented seeded-random behavior — never
  invented;
- raw DatasetNinja `train` -> internal TRAIN + VALIDATION; raw `val` -> academic TEST; raw `test` ->
  **excluded entirely** (0 of 2000 records have any annotated object — no usable ground truth; see
  `docs/decisions_log.md`);
- seed = 42 everywhere.


## 14. Domain-shift limitations (BDD100K -> Colombia)

Hypotheses to validate once Colombian NeuroDriver data is available (never assumed true here):

- different vehicle mix (higher motorcycle density expected in Colombia);
- different road infrastructure/signage;
- different pedestrian behavior/context;
- different weather/lighting distribution;
- different camera mounting/field of view.

See `docs/colombian_domain_strategy.md`.


## 15. Experimental subset

Loads `reports/sampling_plan.md` (produced by `scripts/04_build_experiment_subset.py`).

In [ ]:
sampling_plan_path = PROJECT_ROOT / "reports" / "sampling_plan.md"
if sampling_plan_path.exists():
    print(sampling_plan_path.read_text(encoding="utf-8"))
else:
    print("PENDING: reports/sampling_plan.md not found yet.")


## 16. Conclusions

- The dataset pipeline (Supervisely parser, ROI/frame labeling, Common Manifest, audit, subset,
  validation) is implemented and unit-tested independently of whether the real `.tar` is physically
  present, and has been verified end-to-end against a synthetic Supervisely fixture.
- **Dataset availability in this environment:** see the environment-check output from
  `scripts/00_check_environment.py` and the PENDING markers above.
- Full Frame vs ROI has **not** been permanently chosen; that decision is recorded in
  `docs/decisions_log.md` once real distributions/examples exist.
- Classification is trained as multi-label (has_vehicle, has_pedestrian), not 4-class softmax — see
  Section 6 and `docs/decisions_log.md` (2026-09-22).
- Next notebook (`01_cnn_baseline_mobilenetv2.ipynb`) builds the simple CNN baseline (Student 2), the
  KD-ready MobileNetV2 Student (Student 1), and the future ResNet50 Teacher architecture — still no
  full training in this milestone.
